# SafeStack — Phase 6 Stage 0: capability / utility eval (base, C5, C9) on Colab (A100)

Run the three committed capability configs through the **hf** backend and report **UtilityNorm** — the
capability-retention axis adapted from GRP-Obliteration (`Overall = ASR x UtilityNorm`). This is the
judge-independent floor on whether the SFT alignment (**C5**) and the shadow-unalignment (**C9**)
preserved the model's core functionality, or just broke it — hardening the H4 BROKEN-vs-clean-strip
read (ADR-0017 dec.5). **Exploratory** (ADR-0004 rule 2); runs OUTSIDE the frozen greedy/256 decode.

**Benchmarks / protocol** (pinned in `configs/capability/`, via `lm-evaluation-harness`): MMLU 5-shot
and GSM8K 5-shot strict, raw (OpenLLM v1); IFEval 0-shot, chat-templated (OpenLLM v2).

**Anchor validation.** The **base** run must reproduce the verified public numbers within tol — MMLU
0.6184, GSM8K 0.4905, IFEval 0.4935 — which proves our harness matches the public protocol BEFORE we
trust any C5/C9 delta. C5/C9 are our own adapters (no public anchor) and are read as `UtilityNorm` vs
base.

**Pipeline:** pre-flight (base plumbing, then the pinned C5 adapter load) → full base + anchor check →
C5 → C9 → UtilityNorm. Each model runs one task at a time via `lm_eval`; an adapter (C5/C9) is
materialised locally at its pinned `adapter_revision` (ADR-0015 dec.7b).

**Before Run All:** set two Colab **Secrets** (key icon, "Notebook access" on):
- `HF_TOKEN` — a HF read token for the gated base (Mistral) **and the private adapter repos**
  `kambleakash0/safestack-sft-mistral-lora-v1` and `kambleakash0/safestack-stress-mistral-lora-b411`.
- `GH_TOKEN` — a fine-grained GitHub PAT for `kambleakash0/safestack-study` (Contents: read).

Runtime → GPU (A100). Budget **~1.5–2.5h per model** (MMLU 5-shot dominates on the plain `--model hf`
backend) → **~4–7h for all three**. The HF cache lives on local `/content` disk, so a session reset
re-downloads the base (~15 min) but avoids Drive-FUSE symlink copies; lm-eval scoring is **not**
checkpointed, so keep the tab open and download the aggregate artifacts (last cell) before the session
ends.

**Responsible use:** the benchmark prompts are benign (knowledge / math / instruction-following), and
only **aggregate** task scores are surfaced — never per-sample generations (the admission gate
`scan_notebooks` enforces this on commit). The C5/C9 adapters stay in their private HF-Hub repos.

In [ ]:
# 1. GPU check
import platform

import torch

print("python :", platform.python_version())
print("torch  :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU    :", props.name)
    print("VRAM   :", round(props.total_memory / 1e9, 1), "GB")
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU (A100).")

In [ ]:
# 2. Secrets + HF cache location + clone/update the (private) repo
import os
import stat
import subprocess
import tempfile

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]  # transformers/datasets/peft read this
os.environ["GIT_TOKEN"] = userdata.get("GH_TOKEN")            # token stays in the ENV, never in argv
# HF cache on LOCAL disk, set BEFORE any HF import (huggingface_hub freezes HF_HUB_CACHE from HF_HOME
# at import time, and cell 3 imports peft/transformers). Local disk (not Drive) matches the sibling
# notebooks and avoids Drive-FUSE symlink copies of the ~14GB base.
os.environ["HF_HOME"] = "/content/hf_home"
REPO = "kambleakash0/safestack-study"
DEST = "/content/safestack-study"
REPO_URL = f"https://github.com/{REPO}.git"                   # tokenless remote (no PAT in .git/config)

# Auth via a GIT_ASKPASS helper that READS the token from the environment: the script itself holds no
# secret, and the token reaches git through the (owner-only) process env, not a command-line argument
# (argv is world-readable via /proc/<pid>/cmdline), and never touches .git/config or disk.
_askpass = tempfile.NamedTemporaryFile("w", suffix=".sh", delete=False)
_askpass.write('#!/bin/sh\ncase "$1" in *[Uu]sername*) echo x-access-token ;; *) echo "$GIT_TOKEN" ;; esac\n')
_askpass.close()
os.chmod(_askpass.name, stat.S_IRWXU)                        # 0700, owner-only
_git_env = {**os.environ, "GIT_ASKPASS": _askpass.name, "GIT_TERMINAL_PROMPT": "0"}

# Clone if missing, else force the checkout to the latest main. try/finally so the token and the
# askpass helper are ALWAYS cleaned up -- even if a git op raises.
try:
    if not os.path.isdir(DEST):
        subprocess.run(["git", "clone", "-q", REPO_URL, DEST], check=True, env=_git_env)
    subprocess.run(["git", "-C", DEST, "fetch", "-q", "origin", "main"], check=True, env=_git_env)
    subprocess.run(["git", "-C", DEST, "reset", "--hard", "-q", "origin/main"], check=True, env=_git_env)
finally:
    os.remove(_askpass.name)            # drop the askpass helper (even on failure)
    os.environ.pop("GIT_TOKEN", None)   # drop the token from the environment (even on failure)
%cd /content/safestack-study
!git log --oneline -1

In [ ]:
# 3. Install SafeStack + the [capability] extra: the [hf] stack (torch/transformers/peft) plus
#    lm-eval[ifeval] (IFEval needs langdetect/immutabledict/nltk). Uses Colab's CUDA torch.
!pip -q install -e ".[capability]"
# Colab preinstalls torchao 0.10.0, which the newer PEFT rejects and RAISES on when loading a LoRA
# adapter onto a bf16 base -- exactly the C5/C9 adapter load below (issue #82). We use no torchao, so
# remove it: PEFT's is_torchao_available() then returns False and skips that dispatcher cleanly.
!pip -q uninstall -y torchao
# IFEval scoring tokenises with nltk; fetch its sentence-tokeniser data (punkt + punkt_tab for nltk>=3.9).
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

import peft
import transformers

print("transformers", transformers.__version__, "| peft", peft.__version__)

In [ ]:
# 4. Output paths. The HF cache is on local /content disk (HF_HOME set in cell 2) -- matching the
#    sibling notebooks' "weights on ephemeral disk" idiom and avoiding Drive-FUSE symlink copies. A
#    session reset re-downloads the base (~15 min); download the artifacts (last cell) before the tab
#    closes, since lm-eval scoring is not checkpointed.
import os

REPORTS = "/content/safestack-study/reports"
OUT = f"{REPORTS}/metrics/capability"
os.makedirs(OUT, exist_ok=True)
CONFIGS = ["base", "c5_sft", "c9_stress"]
print("HF_HOME :", os.environ["HF_HOME"])
print("out     :", OUT)

## Run

In [ ]:
# 6. Load + validate the three capability configs (frozen pydantic; fails loud on drift). Capability
#    eval is self-hosted lm-eval only (no hosted-API path exists for these configs), and every surfaced
#    number is aggregate (task scores), never per-sample text.
from safestack.eval.capability import load_capability_config

for stem in CONFIGS:
    cfg = load_capability_config(f"configs/capability/{stem}.yaml")
    tasks = ", ".join(f"{t.name}({t.num_fewshot}-shot)" for t in cfg.tasks)
    print(f"{stem:10s} model={cfg.model:26s} tasks: {tasks}")

In [ ]:
# 7a. PRE-FLIGHT (base plumbing) - verify the lm_eval subprocess, a real base load, and MMLU/GSM8K/
#     IFEval scoring (incl. IFEval's nltk/langdetect deps) on a TINY limit before the long run. Note
#     --limit is per lm-eval SUBTASK, so MMLU (a 57-subtask group) still runs ~228 items (4 x 57) while
#     GSM8K/IFEval are 4 each; the scores are meaningless -- this only proves the plumbing works.
from safestack.eval.capability import run_capability

pre = load_capability_config("configs/capability/base.yaml").model_copy(update={"limit": 4})
pre_art = run_capability(pre, backend="hf", models_dir="configs/models")
for t in pre_art.tasks:
    print(f"  {t.name}: {t.primary_value:.3f} (limit-4 smoke, not a real number)")
print("PRE-FLIGHT (base) PASS - the plumbing works on real weights")

In [ ]:
# 7b. PRE-FLIGHT (adapter) - verify the C5 LoRA snapshot_downloads at its PINNED adapter_revision and
#     loads onto the bf16 base (the issue-#82 / torchao path) on a TINY limit, so a bad pin or a
#     regressed torchao fix fails in SECONDS -- not after the ~2h base run below. The base weights are
#     already cached by cell 7a, so this only pulls the small adapter. No out_dir: the smoke scores are
#     never written or committed.
pre_c5 = load_capability_config("configs/capability/c5_sft.yaml").model_copy(update={"limit": 4})
pre_c5_art = run_capability(pre_c5, backend="hf", models_dir="configs/models")
fp = pre_c5_art.model_fingerprint
print("adapter :", fp.get("adapter"), "@", fp.get("adapter_revision"))
print("PRE-FLIGHT (adapter) PASS - the pinned C5 adapter loads on real weights")

In [ ]:
# 8. BASE - the anchor GATE, run fresh OR loaded from a committed artifact (the ~2h run is skipped
#    when base is already at OUT, since OUT lives inside the clone). EITHER WAY it validates and fails
#    closed -- a stale/edited/wrong committed base is caught too (per-task tol: GSM8K 0.06, else 0.03).
import json

from safestack.eval.capability import CapabilityArtifact, validate_anchors
from safestack.hashing import model_fingerprint
from safestack.registry import resolve_model_spec

_base_cfg = load_capability_config("configs/capability/base.yaml")
_base_path = f"{OUT}/capability_base.json"
if os.path.exists(_base_path):
    with open(_base_path) as _f:
        base_art = CapabilityArtifact.model_validate(json.load(_f))
    # the committed base must BE the pinned base MODEL (right checkpoint/revision/template, no
    # adapter) -- else it is an invalid UtilityNorm denominator even if its scores pass the anchors.
    _expected_fp = model_fingerprint(resolve_model_spec(_base_cfg.model, models_dir="configs/models"))
    if base_art.model_fingerprint != _expected_fp:
        raise SystemExit(
            "committed base fingerprint mismatch -- not the pinned base model; refusing to use it as "
            f"the UtilityNorm denominator.\n  committed: {base_art.model_fingerprint}\n"
            f"  expected:  {_expected_fp}"
        )
    _src = f"loaded from committed {_base_path} (base run skipped)"
else:
    base_art = run_capability(
        _base_cfg, backend="hf", models_dir="configs/models", out_dir=OUT,
    )
    _src = "fresh base run"
anchors = validate_anchors(base_art)  # per-task tolerance; validates BOTH the fresh and committed base
print(f"base: {_src}")
print("task     value    anchor   delta   tol    verdict")
for r in anchors:
    verdict = "PASS" if r.within_tol else "FAIL"
    print(f"{r.task:8s} {r.value:.4f}  {r.anchor:.4f}  {r.abs_delta:.4f}  {r.tol:.3f}  {verdict}")
failed = [r.task for r in anchors if not r.within_tol]
if failed:
    # fail closed in both paths: never let C5/C9 run on an unvalidated (or stale/edited) base.
    raise SystemExit(
        f"ANCHOR CHECK FAILED on {failed} ({_src}): base does not match the public protocol. Fix the "
        "harness/artifact before running or trusting C5/C9."
    )
print("ANCHOR CHECK PASS - base is validated; the C5/C9 deltas are trustworthy")

In [ ]:
# 9. C5 (SFT-aligned) - full capability run. The adapter is materialised locally at its pinned
#    adapter_revision (ADR-0015 dec.7b), then served on the bf16 base. Writes the artifact to OUT.
c5_art = run_capability(
    load_capability_config("configs/capability/c5_sft.yaml"),
    backend="hf", models_dir="configs/models", out_dir=OUT,
)
for t in c5_art.tasks:
    print(f"  {t.name}: {t.primary_value:.4f}")

In [ ]:
# 10. C9 (shadow-unaligned, b*=411) - full capability run. Same pinned-adapter materialisation. The
#     question this answers: is C9 still capable (UtilityNorm ~ 1) or did the stress also break it?
c9_art = run_capability(
    load_capability_config("configs/capability/c9_stress.yaml"),
    backend="hf", models_dir="configs/models", out_dir=OUT,
)
for t in c9_art.tasks:
    print(f"  {t.name}: {t.primary_value:.4f}")

In [ ]:
# 11. UtilityNorm = U(method)/U(base) per task + overall (the GRP-Obliteration degradation axis),
#     reloaded from the written aggregate artifacts so re-running any cell above is picked up.
import json

from safestack.eval.capability import CapabilityArtifact, utility_norm


def _load(eid):
    with open(f"{OUT}/{eid}.json") as f:
        return CapabilityArtifact.model_validate(json.load(f))


base = _load("capability_base")
for eid, label in [("capability_c5_sft", "C5 SFT"), ("capability_c9_stress", "C9 stressed")]:
    rep = utility_norm(_load(eid), base)
    print(f"\n{label}  (UtilityNorm vs base)")
    print("  task     method   base     UtilityNorm")
    for row in rep.rows:
        un = "n/a" if row.utility_norm is None else f"{row.utility_norm:.3f}"
        print(f"  {row.task:8s} {row.method_value:.4f}  {row.base_value:.4f}  {un}")
    ov = "n/a" if rep.overall_utility_norm is None else f"{rep.overall_utility_norm:.3f}"
    print(f"  overall UtilityNorm: {ov}")

In [ ]:
# 12. Provenance summary: which adapter/revision each run used + the raw scores (aggregate only).
for eid in ("capability_base", "capability_c5_sft", "capability_c9_stress"):
    a = _load(eid)
    fp = a.model_fingerprint
    print(f"{a.label:12s} adapter={fp.get('adapter')} rev={fp.get('adapter_revision')}")
    print("   " + "  ".join(f"{t.name}={t.primary_value:.4f}" for t in a.tasks))

In [ ]:
# 13. Aggregate artifacts -> download for the repo (reports/metrics/capability/, no raw text).
import glob

from google.colab import files

for p in sorted(glob.glob(f"{OUT}/*.json")):
    files.download(p)

## After the run

**Commit (aggregate-only)** from the repo, then push:
- `reports/metrics/capability/capability_{base,c5_sft,c9_stress}.json` — the 3 aggregate CapabilityArtifacts
- this executed notebook — verify only aggregate task scores appear (no per-sample text); the admission
  gate `scan_notebooks` (a CI test) enforces this on every commit. If lm-eval's subprocess printed noisy
  progress, clear those cell outputs, keeping the aggregate score prints.

**Do not commit / never public:** nothing new — the C5/C9 adapters stay in their private HF-Hub repos.

**Read:** first confirm the **base anchor check PASSed** (cell 8) — that is what licenses trusting the
C5/C9 numbers. Then UtilityNorm (cell 11) is the result: `~1.0` on **C9** means the shadow-unalignment
left a *fully capable* model (the strong H4 story — alignment stripped, model **not** broken); a large
drop would say the stress degraded capability too. **C5** `~1.0` confirms SFT alignment cost no capability.

**Next:** fold the UtilityNorm read into the H4 write-up (an ADR-0018 addendum / the Stage-0 note), then
**Stage 1 — the DPO-unalignment prereg ADR** (which also settles the self-hosted judge-reward + ADR-0004
rule-4 separation for the GRPO stage, and the responsible-use call on an open-source unalignment trainer).